In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module='jupyter_client')

# Discovering and Visualizing Topics in Texts

Topic models είναι μια οικογένεια μεθόδων που μας βοηθούν να εντοπίσουμε επαναλαμβανόμενα θεματικά μοτίβα μέσα σε ένα μεγάλο σύνολο κειμένων. Η βασική ιδέα είναι ότι λέξεις που εμφανίζονται συχνά μαζί πιθανόν να σχετίζονται με το ίδιο θέμα. Έτσι, ένα «θέμα» δεν είναι μια έτοιμη κατηγορία που δίνουμε εμείς στον υπολογιστή, αλλά ένα σύνολο λέξεων που τείνουν να συνυπάρχουν σε πολλά κείμενα. Ένα από τα πιο γνωστά μοντέλα θεμάτων είναι το Latent Dirichlet Allocation (LDA). Το LDA είναι ένα πιθανοκρατικό μοντέλο που υποθέτει ότι κάθε κείμενο αποτελείται από ένα μείγμα διαφορετικών θεμάτων, ενώ κάθε θέμα αντιστοιχεί σε μια κατανομή λέξεων. Για παράδειγμα, ένα θέμα σχετικό με το ποδόσφαιρο μπορεί να δίνει υψηλή πιθανότητα σε λέξεις όπως «προπονητής», «γκολ» ή «πέναλτι», ενώ ένα πολιτικό θέμα μπορεί να περιλαμβάνει συχνότερα λέξεις όπως «εκλογές», «βουλή» ή «κυβέρνηση».
Σημείωση: topic models δεν «καταλαβαίνουν» πραγματικά το νόημα όπως ένας άνθρωπος· αντίθετα, εντοπίζουν στατιστικά μοτίβα συνύπαρξης λέξεων. Η ερμηνεία των θεμάτων παραμένει πάντα ένα ερευνητικό βήμα που απαιτεί κριτική σκέψη και γνώση του περιεχομένου των κειμένων.

In [ ]:
#!pip install gensim

In [ ]:
#!pip install pyLDAvis

In [ ]:
import pandas as pd

# NLP
import nltk
from nltk.corpus import stopwords

# Topic modeling
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

# Visualization
import pyLDAvis
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files

## Data

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/ML_Lesson/Prog_Clean_sm.csv', encoding='utf-8')


In [ ]:
df.columns

In [ ]:
df.head(2)

In [ ]:
df['body'].head(10)

In [ ]:
df['body'][67]

In [ ]:
df['Institute'].unique()

In [ ]:
df.shape

## Προεπεξεργασία Κειμένων (Preprocessing)

Πριν εκπαιδεύσουμε ένα μοντέλο θεμάτων, χρειάζεται να μετατρέψουμε τα κείμενά μας σε μορφή που να μπορεί να επεξεργαστεί ο υπολογιστής. Η διαδικασία αυτή ονομάζεται preprocessing (προεπεξεργασία). Στο παράδειγμά χρησιμοποιούμε τη συνάρτηση simple_preprocess() της βιβλιοθήκης Gensim, η οποία πραγματοποιεί ορισμένα βασικά βήματα καθαρισμού του κειμένου: μετατρέπει όλες τις λέξεις σε πεζά γράμματα (lowercasing), αφαιρεί σημεία στίξης και ειδικούς χαρακτήρες (deacc=True), διατηρεί μόνο λέξεις με μήκος τουλάχιστον δύο χαρακτήρων (min_len=2) και διαχωρίζει το κείμενο σε λέξεις (tokens). Κάθε κείμενο μετατρέπεται σε μια λίστα λέξεων, την οποία μπορούμε να χρησιμοποιήσουμε για την εκπαίδευση του μοντέλου LDA.

In [ ]:
texts = df['no_stop'].fillna('').tolist()
docs = [simple_preprocess(t, deacc=True, min_len=2) for t in texts]


In [ ]:
# check
print(docs[2])

Δημιουργία Bigrams

Σε πολλά κείμενα, ορισμένες λέξεις εμφανίζονται πολύ συχνά μαζί και σχηματίζουν εκφράσεις με ιδιαίτερο νόημα, π.χ. "climate change", "prime minister". Αν το μοντέλο εξετάσει αυτές τις λέξεις ξεχωριστά, μπορεί να χάσει μέρος της σημασιολογικής τους πληροφορίας. Για τον λόγο αυτό χρησιμοποιούμε bigrams, δηλαδή ζεύγη λέξεων που αντιμετωπίζονται ως μία ενιαία μονάδα. Η Gensim μπορεί να εντοπίσει αυτόματα συχνές συνεμφανίσεις λέξεων και να δημιουργήσει νέους όρους όπως: climate_change, artificial_intelligence.


In [ ]:
# δημιουργία bigram μοντέλου
from gensim.models import Phrases

bigram = Phrases(docs, min_count=10) # min_count=10 --> ένα ζεύγος λέξεων πρέπει να εμφανίζεται τουλάχιστον 10 φορές για να θεωρηθεί bigram

# εξετάζουμε κάθε document ξεχωριστά
for idx in range(len(docs)):

    # το bigram[docs[idx]] επιστρέφει το document μαζί με πιθανά bigrams
    for token in bigram[docs[idx]]:

        # τα bigrams περιέχουν "_" π.χ. "climate_change"
        if '_' in token:

            # προσθέτουμε το bigram στο document
            docs[idx].append(token)

In [ ]:
# check
docs[1]

### Δημιουργία Λεξικού και Bag-of-Words Αναπαράστασης

Αφού ολοκληρώσουμε την προεπεξεργασία των κειμένων, το επόμενο βήμα είναι να μετατρέψουμε τις λέξεις σε μορφή που μπορεί να χρησιμοποιήσει το μοντέλο LDA. Η Gensim δημιουργεί πρώτα ένα λεξικό (dictionary), όπου κάθε μοναδική λέξη αντιστοιχίζεται σε έναν μοναδικό αριθμητικό κωδικό (token id) --> το μοντέλο  επεξεργάζεται αριθμούς αντί για ακατέργαστο κείμενο. Στη συνέχεια, κάθε document μετατρέπεται σε μορφή Bag-of-Words (BoW). Στην αναπαράσταση αυτή δεν μας ενδιαφέρει η σειρά των λέξεων, αλλά μόνο ποιες λέξεις εμφανίζονται και πόσες φορές, π.χ. ['economy', 'economy', 'market'] μπορεί να γίνει: [(12, 2), (45, 1)] δηλαδή, η λέξη με id 12 εμφανίζεται 2 φορές, η λέξη με id 45 εμφανίζεται 1 φορά.

In [ ]:
# δημιουργούμε λεξικό από όλα τα documents-> κάθε μοναδική λέξη παίρνει ένα μοναδικό integer id
from gensim.corpora import Dictionary

dictionary = Dictionary(docs)

# Πόσες μοναδικές λέξεις υπάρχουν συνολικά;
print('Number of unique words in original documents:', len(dictionary))


# εδώ αφαιρούμε...
dictionary.filter_extremes(
    no_below=3,    #...πολύ σπάνιες λέξεις (εμφανίζονται σε < 3 documents)
    no_above=0.25 #... πολύ συχνές λέξεις (εμφανίζονται σε >25% των documents)
)

# πόσες λέξεις έμειαν μετά το φιλτράρισμα;
print('Number of unique words after removing rare and common words:', len(dictionary))


# μετατροπή κειμένου σε Bag-of-Words (BoW)- το doc2bow() επιστρέφει token_id και token_count
print("Example representation of document 15:", dictionary.doc2bow(docs[15]))

#### Now we create a bag-of-word representation for each document in the corpus:

In [ ]:
# δημιουργία corpus
# το corpus είναι μια λίστα που περιέχει τη Bag-of-Words (BoW) αναπαράσταση κάθε κειμένου
corpus = [dictionary.doc2bow(doc) for doc in docs]

In [ ]:
#check
print(corpus[20])

βλέπουμε ότι το token 173 εμφανίζεται 2 φορες ενώ το token 185 εμφανίζεται 20 φορές.

## Training

Τώρα μπορούμε να εκπαιδεύουμε το μοντέλο LDA. Το μοντέλο θα προσπαθήσει να εντοπίσει ομάδες λέξεων που εμφανίζονται συχνά μαζί και να τις οργανώσει σε θέματα. Το LDA δεν γνωρίζει από πριν ποια είναι τα θέματα. Εμείς του δίνουμε το επεξεργασμένο corpus, το λεξικό των λέξεων και τον αριθμό θεμάτων που θέλουμε να αναζητήσει.

In [ ]:
import time
from gensim.models import LdaModel
from tqdm import tqdm # για progress bar

In [ ]:
# train
lda_model = LdaModel(
    corpus=corpus,          # Bag-of-Words αναπαράσταση όλων των κειμένων
    id2word=dictionary,     # αντιστοίχιση token id -> λέξη
    num_topics=15,          # πόσα topics ζητάμε από το μοντέλο
    passes=10,              # πόσες φορές θα "δει" το μοντέλο όλο το corpus κατά την εκπαίδευση
    random_state=42,        # για να παίρνουμε τα ίδια αποτελέσματα κάθε φορά
    alpha='auto',      # ελέγχει πόσα topics τείνει να έχει κάθε document-> χαμηλό alpha: κάθε document τείνει να συνδέεται με λίγα topics,
                        #υψηλό alpha: κάθε document τείνει να είναι πιο μεικτό και να περιέχει περισσότερα topics.
    eta='auto'         # ελέγχει πόσες λέξεις τείνει να περιλαμβάνει κάθε topic-> χαμηλό eta: κάθε topic έχει λίγες λέξεις με υψηλή πιθανότητα, άρα είναι πιο “σφιχτό”.
                        #υψηλό eta: κάθε topic κατανέμει πιθανότητα σε περισσότερες λέξεις, άρα είναι πιο “διάχυτο”.
)

## Results

Για να δουμε τι έμαθε το μοντέλο τυπωνουμε μερικές ενδεικτικές λεξεις

In [ ]:
# print topics
for topic, words in lda_model.print_topics():
    print(topic, ":", words)

### Visualize
Ένας καλός τρόπος για να εξετάσουμε τα topics που παρήγαγε το LDA είναι να τα οπτικοποιήσουμε (με pyLDAvis). Η οπτικοποίηση μάς βοηθά να δούμε πόσο συχνό ή «μεγάλο» είναι κάθε topic μέσα στο corpus, πόσο κοντά ή μακριά είναι τα topics μεταξύ τους, ποιες λέξεις είναι πιο χαρακτηριστικές για κάθε topic. Στο γράφημα, κάθε κύκλος αντιστοιχεί σε ένα topic. Μεγαλύτερος κύκλος σημαίνει ότι το topic εμφανίζεται πιο έντονα στο corpus. Όταν δύο κύκλοι είναι κοντά, τα topics έχουν παρόμοιο λεξιλόγιο. Όταν είναι μακριά, είναι πιο διακριτά μεταξύ τους.

In [ ]:
import pyLDAvis
import warnings
from pyLDAvis import gensim_models as gensimvis


In [ ]:
pyLDAvis.enable_notebook()
warnings.filterwarnings("ignore", category=DeprecationWarning)

# συδνέω το object pyLDAvis με το μοντέλο μου (lda_model), το corpus (bag-of-words) και το dictionary (id->token mapping)
vis = gensimvis.prepare(lda_model, corpus, dictionary, sort_topics=False) # το sort_tppics κρατά την ίδια σειρά topics με το Gensim
pyLDAvis.display(vis)

### Compute Peplexity and Coherence
LDA is an unsupervised method, which means we do not know in advance how many topics are best for our corpus. <br>
Topic *coherence* is one of the most common diagnostics used to evaluate topic quality as it illustrates the level of human topic interpretability. It also guide to the choice of num_topics Here we will compute coherence using:
- c_v, a widely used coherence metric that often aligns better with human judgment
- UMass is based on word co-occurrence statistics <br>

We also compute the *perplexity* of our LDA model on the corpus: it is a measure of how well the model predicts the observed data. Lower (more negative) values indicate a better fit to the data. (Note: a lower perplexity does not always correspond to more interpretable topics: it is typically used with coherence scores).


### Επιλογή Αριθμού Topics

Το LDA είναι μια unsupervised μέθοδος, δηλαδή το μοντέλο δεν γνωρίζει από πριν ποια θέματα υπάρχουν στο corpus ούτε πόσα είναι. Επομένως, ένα από τα βασικά ερευνητικά ερωτήματα είναι πόσα topics πρέπει να ζητήσουμε από το μοντέλο;
Δεν υπάρχει μία απόλυτα σωστή απάντηση. Συνήθως δοκιμάζουμε διαφορετικούς αριθμούς topics και αξιολογούμε τα αποτελέσματα τόσο ποσοτικά όσο και ποιοτικά.
Δύο συχνά μέτρα αξιολόγησης είναι το Perplexity και το Coherence

Η perplexity μετρά πόσο καλά το μοντέλο προβλέπει τα δεδομένα. Χαμηλότερες τιμές σημαίνουν συνήθως καλύτερη προσαρμογή, ωστόσο, η perplexity δεν εγγυάται ότι τα topics είναι εύκολα ερμηνεύσιμα από ανθρώπους.

Η coherence μετρά πόσο «νοηματικά συνεκτικές» είναι οι λέξεις ενός topic.
Αν οι σημαντικότερες λέξεις ενός topic εμφανίζονται συχνά μαζί και σχηματίζουν ένα αναγνωρίσιμο θέμα, τότε η coherence είναι υψηλότερη. Για τον λόγο αυτό, η coherence θεωρείται συχνά πιο χρήσιμη για την ερμηνεία των topics (το μέτρο c_v είναι πολυ διαδεδομένο)

In [ ]:
from gensim.models import CoherenceModel

# Υπολογισμός coherence score
coherence_model = CoherenceModel(
    model=lda_model,
    texts=docs,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()

print("Coherence Score:", coherence_score)

In [ ]:
# Υπολογισμός perplexity
print("Perplexity:", lda_model.log_perplexity(corpus))

### Δοκιμή διαφορετικών αριθμών topics

Συνήθως εκπαιδεύουμε πολλά μοντέλα με διαφορετικά num_topics και συγκρίνουμε τα coherence scores. θα χρησιμοποιήσουμε το c_v coherence metric για να μας βοηθήσει στην επιλογή του k.

In [ ]:
from tqdm import tqdm
from gensim.models import CoherenceModel, LdaModel

In [ ]:
def compute_coherence_values(
    dictionary,      # Gensim dictionary
    corpus,          # Bag-of-Words representation
    texts,           # tokenized documents
    limit,           # maximum number of topics
    start=5,         # minimum number of topics
    step=5,          # increment step
    passes=10,       # passes over the corpus
    iterations=50    # iterations per pass
):

    # λίστα για αποθήκευση των LDA models
    model_list = []

    # λίστα για αποθήκευση coherence scores
    coherence_values = []

    # εκπαίδευση πολλαπλών μοντέλων για διαφορετικούς αριθμούς topics
    for num_topics in tqdm(
        range(start, limit, step),
        desc="Training LDA Models",
        unit="model"
    ):

        # εκπαίδευση LDA
        model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            passes=passes,
            iterations=iterations,
            random_state=1
        )

        model_list.append(model)

        # υπολογισμός coherence score
        coherence_model = CoherenceModel(
            model=model,
            texts=texts,
            dictionary=dictionary,
            coherence='c_v'
        )

        coherence_values.append(
            coherence_model.get_coherence()
        )

    return model_list, coherence_values

In [ ]:
#  δοκιμή διαφορετικών αριθμών topics

model_list, coherence_values = compute_coherence_values(
    dictionary=dictionary,
    corpus=corpus,
    texts=docs,
    start=5,
    limit=30,
    step=5,
    passes=3,
    iterations=20
)

In [ ]:
start = 5
limit = 30
step = 5

In [ ]:
x = list(range(start, limit, step))

df_elbow = pd.DataFrame({
    "Num_Topics": x,
    "Coherence": coherence_values
})

df_elbow

Ας δουμε κι ένα γραφημα του elbow....

In [ ]:
# Plotting tools
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# visualize elbow and coherence scores
limit=30; start=5; step=5;
x = range(start, limit, step)
plt.plot(x, coherence_values)
plt.xlabel("Num Topics")
plt.ylabel("Coherence score")
plt.legend(("coherence_values"), loc='best')
plt.show()

In [ ]:
Το καλυτερο coherence το πετυχαίνουμε γύρω στα 20 topics

## Retrain το τελικό μοντέλο
Mετα την ανάλυση πολλαπλων μοντέλων, διαλέγω το αριθμό topicsμ με το καλυτερο coherence score (στην περίπτωσή μας 20)


In [ ]:
#Retrain
from gensim.models import LdaModel

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=20,
    passes=10,
    random_state=42,
    alpha='auto',
    eta='auto'
)

In [ ]:
# οπτικοποιώ πάλι
from pyLDAvis import gensim_models as gensimvis

pyLDAvis.enable_notebook()
warnings.filterwarnings("ignore", category=DeprecationWarning)

vis = gensimvis.prepare(lda_model, corpus, dictionary, sort_topics=False)
pyLDAvis.display(vis)

#### Print out the topics

In [ ]:
num_topics_to_print = 20  #
num_words_per_topic = 15  #

for (topic, words) in lda_model.print_topics(num_topics=num_topics_to_print, num_words=num_words_per_topic):
    print(f"Topic {topic}: {words}\n")

### Εξέταση των σημαντικότερων λέξεων κάθε topic

Αφού εκπαιδεύσουμε το μοντέλο μας, θα εξετάσουμε ποιες λέξεις θεωρεί το μοντέλο πιο χαρακτηριστικές για κάθε topic. Το LDA αναπαριστά κάθε topic ως μια πιθανότητα πάνω σε όλες τις λέξεις του λεξιλογίου. Αυτό σημαίνει ότι: κάθε λέξη έχει ένα 'βάρος' (weight ή probability) και όσο μεγαλύτερο είναι αυτό το βάρος, τόσο πιο χαρακτηριστική θεωρείται η λέξη για το συγκεκριμένο topic. Παρακατω
επιλέγουμε τις 15 σημαντικότερες λέξεις για κάθε topic, μετατρέπουμε τα word ids πίσω σε πραγματικές λέξεις και αποθηκεύουμε τα αποτελέσματα σε df ώστε να μπορούμε να τα εξετάσουμε πιο εύκολα.

In [ ]:
# πόσες λέξεις θέλουμε να εμφανίζονται ανά topic
n_words = 15

# kενό dataFrame για αποθήκευση αποτελεσμάτων
topic_words = pd.DataFrame()

# επανάληψη πάνω σε όλα τα topics
for i, topic in enumerate(lda_model.get_topics()):

    # το topic είναι ένας πίνακας πιθανοτήτων για όλες τις λέξεις του λεξιλογίου
    # εδώ βρίσκουμε τα ids των λέξεων με τις μεγαλύτερες πιθανότητες
    top_feature_ids = topic.argsort()[-n_words:][::-1]

    # πιθανότητες (weights) των κορυφαίων λέξεων
    feature_values = topic[top_feature_ids]

    # μετατροπή token ids -> πραγματικές λέξεις
    words = [
        dictionary[word_id]
        for word_id in top_feature_ids
    ]

    # προσωρινό DataFrame για το topic
    topic_df = pd.DataFrame({

        "value": feature_values, # βάρος/πιθανότητα λέξης μέσα στο topic
        "word": words, # η πραγματική λέξη
        "topic": i # ο αριθμός topic
    })


    # προσθήκη αποτελεσμάτων στο συνολικό δdtaFrame
    topic_words  = pd.concat([topic_words, topic_df], ignore_index=True)


In [ ]:
#check
topic_words

#### Οπτικοποιουμε την κατανομή των λεξεων

In [ ]:
import seaborn as sns
%matplotlib inline

g = sns.FacetGrid(topic_words, col="topic", col_wrap=5, sharey=False)
g.map(plt.barh, "word", "value")


### Αντιστοίχιση κάθε κειμένου με το κυρίαρχο topic

Μέχρι τώρα εξετάσαμε τα topics ως ομάδες λέξεων. Όμως συχνά θέλουμε να ξέρουμε και κάτι πιο πρακτικό, σε ποιο topic ανήκει κυρίως κάθε κείμενο; Το LDA δεν αναθέτει κάθε κείμενο αποκλειστικά σε ένα μόνο topic, αντίθετα, θεωρεί ότι κάθε κείμενο είναι μια συλλογή από topics. Για παράδειγμα, ένα κείμενο μπορεί να είναι: 60% Topic 2, 25% Topic 5, 15% Topic 1 (ιο αριθμοι είναι τυχαίοι και τους έχει αναθέσει το μοντέλο μόνο του). Παρακάτω κρατάμε το κυρίαρχο topic, δηλαδή το topic με τη μεγαλύτερη πιθανότητα για κάθε κείμενο. Επίσης κρατάμε το ποσοστό συμβολής του και τις βασικές λέξεις του topic (kewywords), ώστε να μπορούμε να ερμηνεύσουμε το αποτέλεσμα.

In [ ]:
def format_topics_sentences(ldamodel, corpus, texts):
    """
    Για κάθε document:
    - βρίσκει το κυρίαρχο topic,
    - κρατά το ποσοστό συμβολής του,
    - κρατά τις βασικές λέξεις του topic,
    - και επιστρέφει τα αποτελέσματα σε DataFrame.
    """

    # lίστα όπου θα αποθηκεύσουμε τα αποτελέσματα μία γραμμή ανά document
    sent_topics_output = []

    # επιστρέφει, για κάθε document τα topics και τις πιθανότητές τους
    for i, row_list in enumerate(ldamodel[corpus]):

        # Σε ορισμένες ρυθμίσεις το LDA επιστρέφει nested output.
        # Αν υπάρχει per_word_topics, κρατάμε μόνο το μέρος
        # που περιέχει τις πιθανότητες των topics.
        row = row_list[0] if ldamodel.per_word_topics else row_list

        # tαξινομούμε τα topics από το πιο πιθανό προς το λιγότερο πιθανό
        row = sorted(row, key=lambda x: x[1], reverse=True)

        # kρατάμε το topic με τη μεγαλύτερη πιθανότητα
        topic_num, prop_topic = row[0]

        # παίρνουμε τις βασικές λέξεις του κυρίαρχου topic
        wp = ldamodel.show_topic(topic_num)

        topic_keywords = ", ".join(
            [word for word, prop in wp]
        )

        # αποθηκεύουμε topic number, ποσοστό συμβολής, λέξεις topic
        sent_topics_output.append([
            int(topic_num),
            round(prop_topic, 4),
            topic_keywords
        ])

    # μετατρέπουμε σε dataFrame
    sent_topics_df = pd.DataFrame(
        sent_topics_output,
        columns=["Dominant_Topic", "Perc_Contribution", "Topic_Keywords"
        ]
    )

    # προσθέτουμε και το αρχικό κείμενο για να μπορούμε να ελέγξουμε την ερμηνεία
    sent_topics_df = pd.concat(
        [sent_topics_df, pd.Series(texts, name="Text")],
        axis=1
    )

    return sent_topics_df


In [ ]:
# εφαρμόζουμε τη συνάρτηση για να βρούμε το κυρίαρχο topic κάθε document
df_topic_sents_keywords = format_topics_sentences(ldamodel=lda_model, corpus=corpus, texts=docs)

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 250)

In [ ]:
df_topic_sents_keywords.head(3)

In [ ]:
# μορφοποίηση του df
df_dominant_topic = df_topic_sents_keywords.reset_index() # kάνω reset index ώστε να έχω και Document_No ως στήλη
df_dominant_topic.columns = ['Document_No', 'Dominant_Topic', 'Topic_Perc_Contrib', 'Keywords', 'Text'] # φιλική ονομασία στηλών

In [ ]:
df_dominant_topic.tail(10)

Looks good...

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 50)

In [ ]:
#check το αρχικό df
df.tail()

### Συγχωνεύω με το αρχικό df

In [ ]:
# merge
df2 = df.merge(df_dominant_topic[["Document_No", "Dominant_Topic", "Topic_Perc_Contrib", "Keywords"]], left_index=True,
    right_on="Document_No", how="left").drop(columns=["Document_No"])

In [ ]:
df2.head()

Το αποτέλεσμα δεν σημαίνει ότι ένα κείμενο “ανήκει” μόνο σε ένα topic. Σημαίνει ότι αυτό είναι το topic με τη μεγαλύτερη συμμετοχή στο συγκεκριμένο κείμενο. Για αυτό κοιτάμε και το Perc_Contribution: όσο μεγαλύτερο είναι, τόσο πιο καθαρά συνδέεται το κείμενο με το συγκεκριμένο topic.

### Ποια topics κυριαρχούν συνολικά;

In [ ]:
topic_counts = (
    df_topic_sents_keywords["Dominant_Topic"]
    .value_counts()
    .sort_index()
)

topic_counts.plot(kind="bar")

plt.xlabel("Topic")
plt.ylabel("Number of Documents")

plt.title("Distribution of Dominant Topics")

plt.show()

### ας φτιάξουμε μερικά labels για τα topics μας...

In [ ]:
# είχαμε μετρήσει από πριν το βάρος κάθε λέξης μέσα σε κάθε topic
topic_words.head(20)

In [ ]:
# κρατάμε τις 3 πρώτες λέξεις με το μεγαλύτερο value για κάθε topic
topic_labels = (
    topic_words
    .groupby("topic")["word"]
    .apply(lambda x: " / ".join(x.head(3)))
    .reset_index()
)

# μετονομάζουμε τη στήλη
topic_labels.columns = [
    "Dominant_Topic",
    "Topic_Label"
]

topic_labels.head()

In [ ]:
df2 = df2.merge(topic_labels, on="Dominant_Topic", how="left")

In [ ]:
df2.head(5)

### Ποια topics κυριαρχούν συνολικά; (με label)

In [ ]:
topic_counts2 = (
    df2["Topic_Label"]
    .value_counts()
    .sort_index()
)
topic_counts2

In [ ]:
topic_counts2.plot(kind="bar")

plt.xlabel("Topic_Label")
plt.ylabel("Number of Documents")

plt.title("Distribution of Dominant Topics")

plt.show()

### Topic evolution over time

In [ ]:
top_topics = df2["Topic_Label"].value_counts().head(8).index

topic_year = pd.crosstab(
    df2["year"],
    df2["Topic_Label"]
)

topic_year[top_topics].plot(figsize=(14, 10))

plt.title("Top 5 Topic Trends Over Time")
plt.xlabel("Year")
plt.ylabel("Number of Documents")
plt.legend(title="Topic", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

### Ποια topics είναι “καθαρά” και ποια πιο mixed;

In [ ]:
df2.groupby("Topic_Label")[
    "Topic_Perc_Contrib"
].mean().sort_values(ascending=False)

In [ ]:
df2.groupby("Topic_Label")[
    "Topic_Perc_Contrib"
].mean().plot(kind="bar")

plt.title("Average Topic Contribution")

plt.ylabel("Average Contribution")

plt.show()

### Συνολικο inspection

In [ ]:
topic_labels_df = (
    df2.groupby("Dominant_Topic")
    .agg({
        "Keywords": "first",
        "Topic_Perc_Contrib": "mean",
        "title": "count"
    })
    .rename(columns={
        "title": "Document_Count",
        "Topic_Perc_Contrib": "Avg_Contribution"
    })
)

topic_labels_df

### Top keywords ανά topic

In [ ]:
df_topics = (
    df2.groupby("Topic_Label")["Keywords"]
    .first()
)

for topic, words in df_topics.items():
    print(f"\nTOPIC {topic}")
    print(words)

In [ ]:
# ποιο think tank ασχολείτai περισσότερο με ποιο topic
pd.crosstab(
    df2["Topic_Label"],
    df2["Institute"]
)

In [ ]:
pd.crosstab(
    df2["Topic_Label"],
    df2["Institute"],
    normalize="columns"
)

### Wordclouds

In [ ]:
topic_words

In [ ]:
from wordcloud import WordCloud

# loop πάνω στα topics
for topic_id in topic_words["topic"].unique():

    # κράταμε μόνο τις λέξεις του συγκεκριμένου topic
    topic_df = topic_words[
        topic_words["topic"] == topic_id
    ]

    # dictionary: word -> weight
    freq_dict = dict(
        zip(topic_df["word"], topic_df["value"])
    )

    # δημιουργία wordcloud
    wc = WordCloud(
        background_color="black",
        width=800,
        height=400
    ).generate_from_frequencies(freq_dict)

    # plot
    plt.figure(figsize=(10,5))

    plt.imshow(wc)

    plt.axis("off")

    plt.title(f"Topic {topic_id}")

    plt.show()

In [ ]:
#df2 = df2.sort_values(['Dominant_Topic', 'Topic_Perc_Contrib'], axis=0).groupby('Dominant_Topic').tail(10)

In [ ]:
# σωζω
df2.to_csv("/my_path/Topics_Prog.csv", index = False)